# FIT3182 Assignment 2 — Data Design & Streaming Application

**Unit:** FIT3182 Big Data Management and Processing — Semester 1, 2026

---

## Table of Contents
1. [Task 1: MongoDB Data Model](#task1)
   - 1.1 Collection Design
   - 1.2 Collection Relationships
2. [Task 2.1.1: Kafka Producers](#task211)
3. [Task 2.1.2: Spark Structured Streaming Join Logic](#task212)
4. [Task 2.1.3: MongoDB Sink Integration](#task213)
5. [Task 2.1.4: Violation Detection](#task214)
6. [Task 3: Documentation & Narrative](#task3)


---
## Step 0: Inspect the Data

Before building the pipeline, we examine the CSV files to understand schemas, data types,
batch structure, timestamp formats, and cross-file relationships.


In [ ]:
# Everything below can be removed due to being exploratory code

# ============================================================
# Data Inspection
# ============================================================
import pandas as pd

# --- vehicle.csv ---
vehicles_df = pd.read_csv("../data/vehicle.csv")
print("=== vehicle.csv ===")
print(f"Shape: {vehicles_df.shape}")
print(f"Columns: {list(vehicles_df.columns)}")
print(f"Duplicate car_plates: {vehicles_df['car_plate'].duplicated().sum()}")
print(vehicles_df.head(3))
print()

# --- camera.csv ---
cameras_df = pd.read_csv("../data/camera.csv")
print("=== camera.csv ===")
print(cameras_df.to_string(index=False))
print()

# --- camera events ---
for label in ['A', 'B', 'C']:
    ev = pd.read_csv("../data/camera_event_{label}.csv", nrows=5)
    full = pd.read_csv("../data/camera_event_{label}.csv")
    print(f"=== camera_event_{label}.csv ===")
    print(f"  Rows: {len(full)}, Columns: {list(ev.columns)}")
    print(f"  Camera IDs: {full['camera_id'].unique()}")
    print(f"  Batch ID range: {full['batch_id'].min()} - {full['batch_id'].max()}")
    print(f"  Timestamp range: {full['timestamp'].min()} to {full['timestamp'].max()}")
    print(f"  Speed range: {full['speed_reading'].min()} - {full['speed_reading'].max()}")
    print()

# --- historic ---
hist = pd.read_csv("../data/camera_event_historic.csv")
print("=== camera_event_historic.csv ===")
print(f"Shape: {hist.shape}")
print(f"Columns: {list(hist.columns)}")
print(hist.head(3))


### Data Inspection Findings

| File | Key Observation |
|------|----------------|
| `vehicle.csv` | 10,000 vehicles; columns: `car_plate, owner_name, owner_addr, vechicle_type, registration_date`; no duplicate plates |
| `camera.csv` | 3 cameras; Camera 1 (pos 152.5, limit 110), Camera 2 (pos 153.5, limit 110), Camera 3 (pos 154.5, limit 90) |
| `camera_event_A.csv` | ~556K events, **all camera_id = 1**, batch_id 1–27817, 20 events per batch |
| `camera_event_B.csv` | ~556K events, **all camera_id = 2**, batch_id 1–294206, variable batch sizes |
| `camera_event_C.csv` | ~556K events, **all camera_id = 3**, batch_id 1–414871, variable batch sizes |
| `camera_event_historic.csv` | 50,000 historical violation records with segment pairs |

**Key observations:**
- Each event file corresponds to **exactly one camera** (A→cam1, B→cam2, C→cam3).
- Timestamps are ISO format strings. Events within a batch share similar timestamps.
- Speed readings are numeric doubles. Some exceed camera limits (violations).
- The segment distance between consecutive cameras is **1.0 km** (153.5 − 152.5 = 1.0, 154.5 − 153.5 = 1.0).


---
<a id="task1"></a>
# Task 1: MongoDB Data Model (2 marks)

## Task 1.1 — Collection Design

We design **four** MongoDB collections: `vehicles`, `cameras`, `violations`, and `camera_events_historic`.

---

### 1.1.1 `vehicles` Collection

**Purpose:** Store vehicle master data — ownership and registration details used for violation lookups.

**Schema:**
```json
{
    "car_plate": "string (unique)",
    "owner_name": "string",
    "owner_addr": "string",
    "vehicle_type": "string",
    "registration_date": "ISODate"
}
```

**Sample Document:**
```json
{
    "car_plate": "FT 02",
    "owner_name": "Goh Mei Wei",
    "owner_addr": "943 Jalan Bukit Mawar, Kuala Lumpur",
    "vehicle_type": "Coupe",
    "registration_date": ISODate("2006-08-22T03:18:00Z")
}
```

**Indexes:**
| Index | Purpose |
|-------|---------|
| `{ "car_plate": 1 }` (unique) | Primary lookup key for vehicle information |

**Shard Key:** `{ "car_plate": "hashed" }` — distributes evenly across shards since plates are unique and unordered.

**Retention Policy:** Permanent — vehicle registration data retained indefinitely for enforcement lookups.

---

### 1.1.2 `cameras` Collection

**Purpose:** Store static camera metadata — location (GeoJSON), position on road, and speed limit.

**Schema:**
```json
{
    "camera_id": "int",
    "location": {
        "type": "Point",
        "coordinates": ["longitude (double)", "latitude (double)"]
    },
    "position": "double (km marker)",
    "speed_limit": "int (km/h)"
}
```

**Sample Document:**
```json
{
    "camera_id": 1,
    "location": {
        "type": "Point",
        "coordinates": [102.6601002, 2.157730731]
    },
    "position": 152.5,
    "speed_limit": 110
}
```

**Indexes:**
| Index | Purpose |
|-------|---------|
| `{ "camera_id": 1 }` (unique) | Fast lookup during streaming join |
| `{ "location": "2dsphere" }` | Geospatial queries for camera proximity |

**Shard Key:** Not sharded — only 3 documents; entire collection fits in memory.

**Retention Policy:** Permanent — camera metadata is static reference data.

---

### 1.1.3 `violations` Collection

**Purpose:** Store detected speed violations. Each document = one car's violations on a **single day**.
Multiple violation events for the same car+day are embedded as an array (daily merging).

**Schema:**
```json
{
    "car_plate": "string",
    "date": "ISODate (date only, no time)",
    "violations": [
        {
            "violation_type": "INSTANTANEOUS | AVERAGE",
            "camera_id_start": "int",
            "camera_id_end": "int",
            "timestamp_start": "ISODate",
            "timestamp_end": "ISODate",
            "speed_reading": "double",
            "speed_limit": "int",
            "excess_kmh": "double"
        }
    ],
    "violation_count": "int",
    "created_at": "ISODate",
    "updated_at": "ISODate"
}
```

**Sample Document:**
```json
{
    "car_plate": "CJW 924",
    "date": ISODate("2024-01-01T00:00:00Z"),
    "violations": [
        {
            "violation_type": "INSTANTANEOUS",
            "camera_id_start": 1,
            "camera_id_end": 1,
            "timestamp_start": ISODate("2024-01-01T08:00:01Z"),
            "timestamp_end": ISODate("2024-01-01T08:00:01Z"),
            "speed_reading": 148.3,
            "speed_limit": 110,
            "excess_kmh": 38.3
        },
        {
            "violation_type": "AVERAGE",
            "camera_id_start": 1,
            "camera_id_end": 2,
            "timestamp_start": ISODate("2024-01-01T08:00:01Z"),
            "timestamp_end": ISODate("2024-01-01T08:00:26Z"),
            "speed_reading": 144.0,
            "speed_limit": 110,
            "excess_kmh": 34.0
        }
    ],
    "violation_count": 2,
    "created_at": ISODate("2024-01-01T08:00:26Z"),
    "updated_at": ISODate("2024-01-01T08:00:26Z")
}
```

**Indexes:**
| Index | Purpose |
|-------|---------|
| `{ "car_plate": 1, "date": 1 }` (unique, compound) | Primary upsert key; merges same-day violations |
| `{ "date": 1 }` | Range queries for daily/weekly reports |
| `{ "violations.violation_type": 1 }` | Filter by violation type |

**Shard Key:** `{ "car_plate": "hashed" }` — distributes write load evenly.

**Retention Policy:** 7 years — aligns with traffic enforcement requirements.

---

### 1.1.4 `camera_events_historic` Collection

**Purpose:** Archive of historical violation records imported from `camera_event_historic.csv`.

**Schema:**
```json
{
    "violation_id": "string (UUID)",
    "car_plate": "string",
    "camera_id_start": "int",
    "camera_id_end": "int",
    "timestamp_start": "ISODate",
    "timestamp_end": "ISODate",
    "speed_reading": "double"
}
```

**Indexes:**
| Index | Purpose |
|-------|---------|
| `{ "car_plate": 1, "timestamp_start": -1 }` | Query vehicle violation history |

**Retention Policy:** Permanent archive.

---

## Task 1.2 — Collection Relationships

### Relationship Diagram

```
┌──────────────┐                    ┌──────────────────┐
│   vehicles   │                    │    violations     │
│              │   referenced by    │                   │
│  car_plate ──┼──────────────────>│  car_plate        │
│  owner_name  │   (via car_plate)  │  date             │
│  vehicle_type│                    │  violations[ ]  ◄─── embedded array
└──────────────┘                    │    violation_type  │
                                    │    camera_id_start │
┌──────────────┐   referenced by    │    camera_id_end   │
│   cameras    │──────────────────>│    speed_reading   │
│  camera_id   │  (camera_id in     │    speed_limit     │
│  speed_limit │   violation record)└──────────────────┘
└──────────────┘
```

### Design Justification: Embed vs. Reference

| Relationship | Strategy | Rationale |
|---|---|---|
| `vehicles` ↔ `violations` | **Reference** (via `car_plate`) | Vehicles are updated independently. Embedding violations inside vehicles would cause unbounded document growth (a vehicle may accumulate thousands of violations over years). Referencing keeps writes isolated and collections independently queryable. The trade-off is an extra lookup when displaying owner info, but this is infrequent and cheap. |
| `cameras` ↔ `violations` | **Reference** (camera_id stored in violation sub-record) | Camera data is static and tiny (3 docs). Embedding full camera info in every violation would duplicate data unnecessarily. A lookup at query time is cheap for 3 documents. |
| Daily violation sub-records | **Embedded** (array in daily document) | Multiple violations for the same car on the same day are logically grouped and always retrieved together. Embedding avoids joins and enables atomic `$push` upserts. The array is naturally bounded — a car can only trigger a limited number of violations per day (max ~6: instant at each of 3 cameras + avg on 2 segments). |

### Read/Write Pattern Analysis
- **Writes (high frequency):** The `violations` collection receives streaming upserts. The compound key `{ car_plate, date }` enables efficient `update_one(upsert=True)` with `$push` — Mongo finds or creates the daily doc and appends the new violation atomically.
- **Reads (moderate):** Visualisation queries read violations by date range or by plate. The `{ date: 1 }` index supports time-range scans. The compound index supports plate-specific lookups.
- **Consistency:** Using `$push` with the compound unique index ensures that even if a streaming batch is replayed, the document structure remains consistent. For strict idempotency, we can check for duplicate `violation_id` before pushing.


---
<a id="task211"></a>
# Task 2: Streaming Application

## Environment Setup


In [ ]:
# ============================================================
# Imports
# ============================================================
import os
import json
import time
import uuid
import csv
import threading
from datetime import datetime, timedelta

# Kafka
from kafka import KafkaProducer, KafkaConsumer
from kafka.admin import KafkaAdminClient, NewTopic

# Spark
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, from_json, expr, lit, udf, to_timestamp,
    unix_timestamp, current_timestamp, when, date_format, to_date
)
from pyspark.sql.types import (
    StructType, StructField, StringType, IntegerType,
    DoubleType, TimestampType, BooleanType
)

# MongoDB
from pymongo import MongoClient, UpdateOne, ASCENDING, DESCENDING
from pymongo.errors import BulkWriteError

import warnings
warnings.filterwarnings('ignore')

# Consider removing debugging line below
# print("All imports successful.")

## Load Reference Data into MongoDB

Load `vehicle.csv`, `camera.csv`, and `camera_event_historic.csv` into their respective MongoDB collections.


In [ ]:
mg_db_url = "mongodb://localhost:27017/"
db_name = "AWAS_speed_violations"

# Initialise mongodb with new db
client = MongoClient(mg_db_url)
db = client[db_name]

# Drop in case already existing
existing_db = ["vehicles", "cameras", "violations", "camera_events_historic"]
for col in existing_db:
    db.drop_collection(col)

# Consider removing line below
# print(f"Connected to MongoDB database: {db_name}")


In [ ]:
import pandas as pd

# Read from vehicle csv file
vehic_df = pd.read_csv("../data/vehicle.csv")

vehicles = []

# Get vehicle rows
for _, row in vehic_df.iterrows():
    vehicles.append({
        "car_plate": row["car_plate"].strip(),
        "owner_name": row["owner_name"].strip(),
        "owner_addr": row["owner_addr"].strip(),
        "vehicle_type": row["vechicle_type"].strip(),
        "registration_date": datetime.fromisoformat(row["registration_date"])
    })

# Insert rows into collection
db.vehicles.insert_many(vehicles)

# Create index on car plate column
db.vehicles.create_index([("car_plate", ASCENDING)], unique = True, name = "idx_car_plate")

# print(f"Loaded {len(vehicles)} vehicles.")

In [ ]:
# Read from camera csv file
cam_df = pd.read_csv("../data/camera.csv")

cameras = []

# Get camera rows
for _, row in cam_df.iterrows():
    cameras.append({
        "camera_id": int(row["camera_id"]),
        "location": {
            "type": "Point",
            "coordinates": [float(row["longitude"]), float(row["latitude"])]
        },
        "position": float(row["position"]),
        "speed_limit": int(row["speed_limit"])
    })

# Insert rows into collection
db.cameras.insert_many(cameras)

# Create index on camera ID and camera location columns
db.cameras.create_index([("camera_id", ASCENDING)], unique=True, name="idx_camera_id")
db.cameras.create_index([("location", "2dsphere")], name="idx_location_geo")

# Can remove below lines

"""print(f"Loaded {len(cameras)} cameras:")
for doc in cameras:
    print(f"  Camera {doc['camera_id']}: position={doc['position']}km, "
          f"limit={doc['speed_limit']}km/h, coords={doc['location']['coordinates']}")"""

In [ ]:
# Read from historic csv file
hist_df = pd.read_csv("../data/camera_event_historic.csv")

historics = []
hist_cnt = 0

# Get historic rows
for _, row in hist_df.iterrows():
    hist_cnt += 1
    historics.append({
        "violation_id": row["violation_id"].strip(),
        "car_plate": row["car_plate"].strip(),
        "camera_id_start": int(row["camera_id_start"]),
        "camera_id_end": int(row["camera_id_end"]),
        "timestamp_start": datetime.fromisoformat(row["timestamp_start"]),
        "timestamp_end": datetime.fromisoformat(row["timestamp_end"]),
        "speed_reading": float(row["speed_reading"])
    })

step = 5000

# Insert in groups
for i in range(0, hist_cnt, step):
    db.camera_events_historic.insert_many(historics[i : (i + step)])

# Create index on car plate and starting timestamp columns
db.camera_events_historic.create_index(
    [("car_plate", ASCENDING), ("timestamp_start", DESCENDING)],
    name="idx_plate_time"
)

# print(f"Loaded {len(hist_docs)} historic records.")

In [ ]:
# Create index on car plate and date columns
db.violations.create_index(
    [("car_plate", ASCENDING), ("date", ASCENDING)],
    unique=True, name="idx_car_date"
)

# Create index on date column
db.violations.create_index([("date", ASCENDING)], name="idx_date")

# Create index on violation type
db.violations.create_index(
    [("violations.violation_type", ASCENDING)], name="idx_viol_type"
)

# Can consider removing code below

"""for col in existing_db:
    indexes = list(db[col].index_information().keys())
    print(f"{col}: {indexes}")"""

---
<a id="task211"></a>
## Task 2.1.1 — Kafka Producer Implementation

### Design

Each producer reads one camera event CSV file and publishes records to a dedicated Kafka topic,
grouped by `batch_id`. Between batches, the producer sleeps for `BATCH_INTERVAL` seconds to
simulate real-time ingestion.

| Producer | CSV File | Kafka Topic | Camera |
|----------|----------|-------------|--------|
| Producer A | `camera_event_A.csv` | `camera-events-A` | Camera 1 (limit 110) |
| Producer B | `camera_event_B.csv` | `camera-events-B` | Camera 2 (limit 110) |
| Producer C | `camera_event_C.csv` | `camera-events-C` | Camera 3 (limit 90) |

**Metadata enrichment:** Each message includes `producer_id` for traceability and debugging.


In [ ]:
# ============================================================
# Kafka Configuration
# ============================================================
kafka_url = "localhost:9092"
topics = {
    "A": "camera-events-A",
    "B": "camera-events-B",
    "C": "camera-events-C"
}
inter = 2  # seconds between batches (adjust for testing)

# Create topics (idempotent)
try:
    admin = KafkaAdminClient(bootstrap_servers = kafka_url)
    existing = admin.list_topics()
    new_topics = [
        NewTopic(name=t, num_partitions=3, replication_factor=1)
        for t in topics.values() if t not in existing
    ]
    if new_topics:
        admin.create_topics(new_topics)
        print(f"Created topics: {[t.name for t in new_topics]}")
    else:
        print("All topics already exist.")
    admin.close()
except Exception as e:
    print(f"Topic setup note: {e}")

In [ ]:
def produce(file_p, topic, prod_id, interval):
	# Creating Kafka producer
    producer = KafkaProducer(
        bootstrap_servers = kafka_url,
        value_serializer = lambda val : json.dumps(val).encode('utf-8'),
        key_serializer = lambda key : key.encode('utf-8') if key else None,
        linger_ms = 50,
        acks = 'all',
        retries = 3
    )

    batches = {}

    # Read CSV file and group rows by batch ID
    with open(file_p, 'r') as file:
        reader = csv.DictReader(file)
        for row in reader:
            batch_id = int(row['batch_id'])
            batches.setdefault(batch_id, []).append(row)

    # Get sorted batch IDs
    sorted_bids = sorted(batches.keys())

	# Consider removing lines below

    """total_events = sum(len(v) for v in batches.values())
    print(f"[Producer {producer_id}] {len(sorted_bids)} batches, "
          f"{total_events} events from {os.path.basename(csv_path)}")"""

    sent = 0
    for bid in sorted_bids:
        for row in batches[bid]:
            event = {
                "event_id": row["event_id"],
                "batch_id": int(row["batch_id"]),
                "car_plate": row["car_plate"].strip(),
                "camera_id": int(row["camera_id"]),
                "timestamp": row["timestamp"].strip(),
                "speed_reading": float(row["speed_reading"]),
                "producer_id": prod_id
            }

            producer.send(topic, key = event["car_plate"], value = event)
            producer.flush()

            sent += 1

		# Consider removing the lines below

        # Progress log every 500 batches
        """if bid % 500 == 0:
            print(f"  [Producer {producer_id}] batch {bid}/{sorted_bids[-1]} "
                  f"({sent} events)")"""

        time.sleep(interval)

    producer.flush()
    producer.close()

    # print(f"[Producer {producer_id}] Done — {sent} events published.")

    return (sent)


In [ ]:
# Approach here uses locally define threads to produce camera events
# whereas files such as producer_a.ipynb does it externally

producer_params = [
    ("../data/camera_event_A.csv", topics["A"], "A"),
    ("../data/camera_event_B.csv", topics["B"], "B"),
    ("../data/camera_event_C.csv", topics["C"], "C"),
]

threads = []

# Produce 3 camera streams
for file_p, topic, prod_id in producer_params:
    t = threading.Thread(target = produce, args = (file_p, topic, prod_id), daemon = True)
    threads.append(t)
    t.start()
    # print(f"Started Producer {prod_id}")

# Wait for all to finish
for t in threads:
    t.join()

# print("\nAll producers completed.")

---
<a id="task212"></a>
## Task 2.1.2 — Spark Structured Streaming Join Logic

### Architecture

```
                         Kafka                      Spark Structured Streaming
                     ┌────────────┐
camera_event_A.csv → │ camera-    │──→ camera_a_stream ─┐
                     │ events-A   │                      ├─→ joined_AB (A-B pairs)
camera_event_B.csv → │ camera-    │──→ camera_b_stream ─┤
                     │ events-B   │                      ├─→ joined_BC (B-C pairs)
camera_event_C.csv → │ camera-    │──→ camera_c_stream ─┘
                     │ events-C   │
                     └────────────┘

    Violation Detection:
    ├── Instantaneous: speed_reading > camera speed_limit
    ├── Average (A→B): avg_speed > Camera 2 limit (110 km/h)
    └── Average (B→C): avg_speed > Camera 3 limit (90 km/h)
             │
             ▼
        MongoDB: violations collection (foreachBatch sink)
```

### Key Design Decisions
| Parameter | Value | Rationale |
|-----------|-------|-----------|
| Watermark | 10 minutes | Generous buffer for late-arriving events while bounding state |
| Join window | 60 seconds | Max reasonable time to travel 1 km (equivalent to 60 km/h minimum speed) |
| Segment distance | 1.0 km | Derived from camera positions (153.5−152.5 = 1.0, 154.5−153.5 = 1.0) |
| Trigger interval | 10 seconds | Balances latency with micro-batch processing efficiency |
| Join type | Inner | Matches valid camera pairs; unmatched events expire via watermark |


In [ ]:
# Initialise Spark Session
spark = SparkSession.builder \
    .appName("AWAS_Speed_Violitions") \
    .master("local[*]") \
    .config("spark.jars.packages",
            "org.apache.spark:spark-sql-kafka-0-10_2.12:3.5.0") \
    .config("spark.sql.shuffle.partitions", "8") \
    .config("spark.streaming.stopGracefullyOnShutdown", "true") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")

#print(f"Spark version: {spark.version}")

In [ ]:
# Initialise schema for events
event_schema = StructType([
    StructField("event_id", StringType()),
    StructField("batch_id", IntegerType()),
    StructField("car_plate", StringType()),
    StructField("camera_id", IntegerType()),
    StructField("timestamp", StringType()),
    StructField("speed_reading", DoubleType()),
    StructField("producer_id", StringType())
])

# print("Event schema:")
# event_schema.printTreeString()


In [ ]:
# ============================================================
# Read Kafka Streams — one per camera topic
# ============================================================
def read_cam_stream(topic):
    read = spark.readStream \
           .format("kafka") \
           .option("kafka.bootstrap.servers", kafka_url) \
           .option("subscribe", topic) \
           .option("startingOffsets", "latest") \
           .option("failOnDataLoss", "false") \
           .load() \
           .selectExpr("CAST(value AS STRING) AS json_value") \
           .select(from_json(col("json_value"), event_schema).alias("data")) \
           .select("data.*") \
           .withColumn("event_time", to_timestamp(col("timestamp")))

    return (read)

# Read from all camera streams
cam_a_stream = read_cam_stream(topics["A"])
cam_b_stream = read_cam_stream(topics["B"])
cam_c_stream = read_cam_stream(topics["C"])

# Remove lines below

"""print("Kafka streams created:")
print(f"  camera_a_stream → {topics['A']} (Camera 1)")
print(f"  camera_b_stream → {topics['B']} (Camera 2)")
print(f"  camera_c_stream → {topics['C']} (Camera 3)")"""


### Speed Limit Lookup

For 3 cameras, a simple dictionary lookup is efficient and avoids broadcast joins.


In [ ]:
# Defining speed limits for each camera
speed_limits = {
    1: 110,
    2: 110,
    3: 90
}

# Defining distances for camera pair 1 to 2 and
# camera pair 2 to 3
segment_distances = {
    (1, 2): 1.0,
    (2, 3): 1.0
}

# print(f"Speed limits: {speed_limits}")
# print(f"Segment distances: {segment_distances}")

### Instantaneous Violation Detection

A vehicle is flagged when its speed_reading exceeds the recording camera's speed limit.
We use a UDF to check this condition.


In [ ]:
# ============================================================
# Instantaneous Violation Detection (UDF approach)
# ============================================================
def is_speeding(camera_id, speed):
    limit = speed_limits.get(camera_id, float("inf"))

    return (speed > limit)

# Creating the custom function
is_speeding_udf = udf(is_speeding, BooleanType())

# Join all streams together
all_cams = cam_a_stream \
              .union(cam_b_stream) \
              .union(cam_c_stream)

# Filter for instantaneous violations
instant_violations = all_cams.filter(
    is_speeding_udf(col("camera_id"), col("speed_reading"))
)

# print("Instantaneous violation filter configured.")


### Stream-Stream Joins for Average Speed Detection

We perform **two separate joins**: A→B and B→C.

Each join matches the **same car_plate** across consecutive cameras, where:
- The end event occurs **after** the start event
- The end event occurs **within 60 seconds** of the start event
- The watermark (10 minutes) bounds the retained state

Average speed = distance (km) / travel_time (hours)


In [ ]:
# ============================================================
# Apply Watermarks for Stream-Stream Joins
# ============================================================
watermark = "10 minutes"
join_time = "60 seconds"

wm_a = cam_a_stream.withWatermark("event_time", watermark)
wm_b = cam_b_stream.withWatermark("event_time", watermark)
wm_c = cam_c_stream.withWatermark("event_time", watermark)

# print(f"Watermark: {watermark}")
# print(f"Join window: {join_time}")


In [ ]:
joined_AB = (
    wm_a.alias("a")
    .join(
        wm_b.alias("b"),
        expr(f"""
            a.car_plate = b.car_plate
            AND a.event_time < b.event_time
            AND b.event_time <= a.event_time + interval {join_time}
        """),
        "inner"
    )
    .select(
        col("a.car_plate").alias("car_plate"),
        col("a.camera_id").alias("camera_id_start"),
        col("b.camera_id").alias("camera_id_end"),
        col("a.event_time").alias("timestamp_start"),
        col("b.event_time").alias("timestamp_end"),
        col("a.speed_reading").alias("speed_start"),
        col("b.speed_reading").alias("speed_end")
    )
)

# Calculate average speed for A→B segment
dist = segment_distances[(1,2)]  # km
limit = speed_limits[2]  # 110 km/h (ending camera limit)

ab_with_speed = (
    joined_AB
    .withColumn(
        "travel_time_hours",
        (unix_timestamp(col("timestamp_end")) - unix_timestamp(col("timestamp_start")))
        / lit(3600.0)
    )
    .withColumn("avg_speed", lit(dist) / col("travel_time_hours"))
    # Filter: only valid travel times (> 0)
    .filter(col("travel_time_hours") > 0)
)

# Flag average speed violations: avg_speed > ending camera's limit
ab_avg_violations = ab_with_speed.filter(col("avg_speed") > lit(limit))

# Remove lines below

"""print(f"A→B join configured: distance={dist}km, "
      f"ending limit={limit}km/h")"""

In [ ]:
joined_BC = (
    wm_b.alias("b2")
    .join(
        wm_c.alias("c"),
        expr(f"""
            b2.car_plate = c.car_plate
            AND b2.event_time < c.event_time
            AND c.event_time <= b2.event_time + interval {join_time}
        """),
        "inner"
    )
    .select(
        col("b2.car_plate").alias("car_plate"),
        col("b2.camera_id").alias("camera_id_start"),
        col("c.camera_id").alias("camera_id_end"),
        col("b2.event_time").alias("timestamp_start"),
        col("c.event_time").alias("timestamp_end"),
        col("b2.speed_reading").alias("speed_start"),
        col("c.speed_reading").alias("speed_end")
    )
)

# Calculate average speed for B→C segment
dist = segment_distances[(2,3)]  # km
limit = speed_limits[3]  # 90 km/h (ending camera limit)

bc_with_speed = (
    joined_BC
    .withColumn(
        "travel_time_hours",
        (unix_timestamp(col("timestamp_end")) - unix_timestamp(col("timestamp_start")))
        / lit(3600.0)
    )
    .withColumn("avg_speed", lit(dist) / col("travel_time_hours"))
    .filter(col("travel_time_hours") > 0)
)

# Flag average speed violations
bc_avg_violations = bc_with_speed.filter(col("avg_speed") > lit(limit))

# Remove lines below

"""print(f"B→C join configured: distance={dist}km, "
      f"ending limit={limit}km/h")"""


---
<a id="task213"></a>
## Task 2.1.3 — Sink Integration with MongoDB

### Strategy
- Use `foreachBatch` to write each micro-batch to MongoDB.
- Use `update_one(..., upsert=True)` with `$push` to merge same-day violations.
- Compound key: `{ car_plate, date }` ensures daily merging.
- Bulk writes with `ordered=False` for efficiency and partial-failure resilience.
- Idempotency consideration: streaming retries may re-process events; the daily-document
  structure plus unique compound index keeps the document consistent.


In [ ]:
def write_instant_viols(batch_df, batch_id):
    if batch_df.isEmpty():
        return

    rows = batch_df.collect()
    if not rows:
        return

    mongo = MongoClient(mg_db_url)
    viols = mongo[db_name]["violations"]

    operations = []
    for row in rows:
        car_plate = row["car_plate"]
        event_ts = row["event_time"]
        cam_id = row["camera_id"]
        speed = row["speed_reading"]
        limit = speed_limits.get(cam_id, 110)
        violation_date = datetime(event_ts.year, event_ts.month, event_ts.day)

        violation_detail = {
            "violation_type": "INSTANTANEOUS",
            "camera_id_start": cam_id,
            "camera_id_end": cam_id,
            "timestamp_start": event_ts,
            "timestamp_end": event_ts,
            "speed_reading": round(speed, 1),
            "speed_limit": limit,
            "excess_kmh": round(speed - limit, 1)
        }

        operations.append(UpdateOne(
            {"car_plate": car_plate, "date": violation_date},
            {
                "$setOnInsert": {
                    "car_plate": car_plate,
                    "date": violation_date,
                    "created_at": datetime.now()
                },
                "$push": {"violations": violation_detail},
                "$inc": {"violation_count": 1},
                "$set": {"updated_at": datetime.now()}
            },
            upsert=True
        ))

    # Bulk write — ordered=False for partial-failure resilience
    if operations:
        try:
            viols.bulk_write(operations, ordered = False)
        except BulkWriteError as bwe:
            print(f"[Batch {batch_id}] INSTANT partial error: "
                  f"{bwe.details.get('nErrors', 0)} errors")

    mongo.close()

In [ ]:
def make_avg_violation_writer(segment_label, ending_limit):
    def write_avg_viols(batch_df, batch_id):
        if batch_df.isEmpty():
            return

        rows = batch_df.collect()
        if not rows:
            return

        mongo = MongoClient(mg_db_url)
        viol_coll = mongo[db_name]["violations"]

        operations = []
        for row in rows:
            car_plate = row["car_plate"]
            ts_start = row["timestamp_start"]
            ts_end = row["timestamp_end"]
            avg_spd = row["avg_speed"]
            violation_date = datetime(ts_end.year, ts_end.month, ts_end.day)

            violation_detail = {
                "violation_type": "AVERAGE",
                "camera_id_start": row["camera_id_start"],
                "camera_id_end": row["camera_id_end"],
                "timestamp_start": ts_start,
                "timestamp_end": ts_end,
                "speed_reading": round(avg_spd, 1),
                "speed_limit": ending_limit,
                "excess_kmh": round(avg_spd - ending_limit, 1)
            }

            operations.append(UpdateOne(
                {"car_plate": car_plate, "date": violation_date},
                {
                    "$setOnInsert": {
                        "car_plate": car_plate,
                        "date": violation_date,
                        "created_at": datetime.now()
                    },
                    "$push": {"violations": violation_detail},
                    "$inc": {"violation_count": 1},
                    "$set": {"updated_at": datetime.now()}
                },
                upsert=True
            ))

        if operations:
            try:
                result = viol_coll.bulk_write(operations, ordered=False)
                print(f"  [Batch {batch_id}] AVG {segment_label}: "
                      f"{result.upserted_count} new, {result.modified_count} updated")
            except BulkWriteError as bwe:
                print(f"  [Batch {batch_id}] AVG {segment_label} error: "
                      f"{bwe.details.get('nErrors', 0)} errors")

        mongo.close()

    return write_avg_viols

# Create writers for each segment
write_ab_violations = make_avg_violation_writer("A -> B", speed_limits[2])
write_bc_violations = make_avg_violation_writer("B -> C", speed_limits[3])

# Remove lines below

"""print("MongoDB sink writers created:")
print(f"  Instantaneous: write_instant_viols")
print(f"  Average A→B:   write_ab_violations (limit {speed_limits[2]})")
print(f"  Average B→C:   write_bc_violations (limit {speed_limits[3]})")"""

---
<a id="task214"></a>
## Task 2.1.4 — Start Streaming Queries

### Violation Detection Rules
1. **INSTANTANEOUS:** `speed_reading > camera speed_limit` at the recording camera.
2. **AVERAGE (A→B):** `avg_speed = 1.0 km / travel_time > 110 km/h` (Camera 2 limit).
3. **AVERAGE (B→C):** `avg_speed = 1.0 km / travel_time > 90 km/h` (Camera 3 limit).
4. **Daily merging:** `$push` into same document keyed by `{ car_plate, date }`.

### Handling Invalid / Unmatched Records
- **Unmatched events:** If a car appears at Camera 1 but never at Camera 2 within 60 seconds,
  the event expires from Spark's state store (evicted by watermark). This is expected — not
  every car passes all cameras.
- **Invalid timing:** We filter `travel_time_hours > 0` to discard physically impossible pairs.
- **Late events:** The 10-minute watermark allows significant late arrival tolerance.


In [ ]:
checkpoint = "/tmp/awas_checkpoints"

# Query 1: Instantaneous violations (all cameras)
query_instant = (
    instant_violations
    .writeStream
    .foreachBatch(write_instant_viols)
    .option("checkpointLocation", "/tmp/awas_checkpoints/instant")
    .outputMode("append")
    .trigger(processingTime = "10 seconds")
    .start()
)

# print("Started: Instantaneous violation detection")

# Query 2: Average speed violations (A to B)
query_ab = (
    ab_avg_violations
    .writeStream
    .foreachBatch(write_ab_violations)
    .option("checkpointLocation", "/tmp/awas_checkpoints/avg_ab")
    .outputMode("append")
    .trigger(processingTime = "10 seconds")
    .start()
)

# print("Started: Average speed detection A→B")

# Query 3: Average speed violations (B to C)
query_bc = (
    bc_avg_violations
    .writeStream
    .foreachBatch(write_bc_violations)
    .option("checkpointLocation", "/tmp/awas_checkpoints/avg_bc")
    .outputMode("append")
    .trigger(processingTime = "10 seconds")
    .start()
)

# print("Started: Average speed detection B→C")

# print("\nAll 3 streaming queries are running.")

In [ ]:
# ============================================================
# Monitor Streaming Progress
# ============================================================
m_secs = 180  # adjust based on data volume

# print(f"Monitoring for {m_secs}s...")
# print("=" * 60)

for elapsed in range(15, m_secs + 1, 15):
    time.sleep(15)

    # Check query statuses
    queries = [("Instant", query_instant), ("Avg A to B", query_ab), ("Avg B to C", query_bc)]
    for name, q in queries:
        prog = q.lastProgress
        rows = prog.get("numInputRows", 0) if prog else "N/A"
        print(f"  [{name}] rows={rows}")

    # MongoDB stats
    """viol_docs = db.violations.count_documents({})
    print(f"  [MongoDB] violation docs: {viol_docs}")
    print(f"  Elapsed: {elapsed}s / {MONITOR_SECONDS}s")
    print("-" * 40)"""

#print("\nMonitoring complete.")

In [ ]:
# Stops after set amount of time, m_secs
query_instant.stop()
query_ab.stop()
query_bc.stop()
# print("All streaming queries stopped.")

# Final statistics
viol_docs = db.violations.count_documents({})
pipeline = [{"$group": {"_id": None, "total": {"$sum": "$violation_count"}}}]
agg = list(db.violations.aggregate(pipeline))
total_viols = agg[0]["total"] if agg else 0

print(f"\n{'='*50}")
print(f"FINAL RESULTS")
print(f"{'='*50}")
print(f"  Violation documents (car × day): {viol_docs}")
print(f"  Total individual violations:     {total_viols}")

# Show a sample document
import pprint
sample = db.violations.find_one()
if sample:
    print(f"\nSample violation document:")
    pprint.pprint(sample)


---
<a id="task3"></a>
# Task 3: Documentation & Code Quality

## 3.1 Code Comments

All functions include docstrings. Key logic sections (watermarking, join conditions, violation
filtering, MongoDB upsert) have inline comments explaining the reasoning.

## 3.2 Narrative Explanation

### End-to-End Architecture Diagram

```
┌──────────────────────────────────────────────────────────────────┐
│                        DATA SOURCES                              │
├──────────────┬───────────────┬───────────────┬──────────────────┤
│ vehicle.csv  │ camera.csv    │ event_A/B/C   │ historic.csv     │
│ (10K rows)   │ (3 cameras)   │ (~556K each)  │ (50K violations) │
└──────┬───────┴───────┬───────┴───────┬───────┴───────┬──────────┘
       │               │               │               │
       ▼               ▼               │               ▼
  ┌─────────┐    ┌──────────┐         │         ┌──────────────┐
  │vehicles │    │ cameras  │         │         │camera_events │
  │(MongoDB)│    │(MongoDB) │         │         │  _historic   │
  └─────────┘    └──────────┘         │         │ (MongoDB)    │
                                      │         └──────────────┘
                                      ▼
                              ┌───────────────┐
                              │ Kafka         │
                              │ Producers     │
                              │ (3 threads)   │
                              └───────┬───────┘
                                      │
                    ┌─────────────────┼─────────────────┐
                    ▼                 ▼                  ▼
              camera-events-A  camera-events-B    camera-events-C
              (Camera 1)       (Camera 2)         (Camera 3)
                    │                 │                  │
                    ▼                 ▼                  ▼
              ┌─────────────────────────────────────────────┐
              │     Spark Structured Streaming               │
              │                                              │
              │  ┌─────────────────────────────────────┐    │
              │  │ Stream Parsing + Watermark (10 min)  │    │
              │  └──────────────┬───────────────────────┘    │
              │                 │                              │
              │    ┌────────────┼─────────────┐               │
              │    ▼            ▼             ▼               │
              │  Instant    Join A→B      Join B→C            │
              │  Filter    (60s window)  (60s window)         │
              │    │            │             │               │
              │    ▼            ▼             ▼               │
              │  speed>limit  avg>110     avg>90              │
              │    │            │             │               │
              │    └────────────┼─────────────┘               │
              │                 │                              │
              │         foreachBatch sinks                     │
              └─────────────────┬─────────────────────────────┘
                                │
                                ▼
                        ┌───────────────┐
                        │  MongoDB      │
                        │  violations   │──→ Visualisation
                        │  collection   │    (Plotly + Matplotlib)
                        └───────────────┘
```

### Key Parameters

| Parameter | Value | Rationale |
|-----------|-------|-----------|
| Producer batch interval | 2 seconds | Simulates real-time camera event frequency |
| Watermark | 10 minutes | Generous late-event tolerance; bounds state |
| Join window | 60 seconds | Max 1 km travel at ≥60 km/h; filters unreasonable pairs |
| Trigger interval | 10 seconds | Balance between latency and batch efficiency |
| Segment distance | 1.0 km | Calculated from camera.csv positions |
| MongoDB upsert key | `{ car_plate, date }` | Merges same-day violations atomically |

### Assumptions & Limitations
1. Cameras are sequential (1 → 2 → 3) along the road segment.
2. Only forward-direction travel is monitored (camera order must be ascending).
3. Segment distance is fixed at 1.0 km based on camera positions.
4. A 60-second join window filters out physically unreasonable travel times.
5. Events older than the 10-minute watermark are evicted from state.
6. Unmatched events (car at Camera 1 but not Camera 2) are silently dropped.
7. Historical violations from `camera_event_historic.csv` are loaded for reference
   but not re-processed through the streaming pipeline.
